# 🌱 FarmerBud · 2. Which crops should I sow in my field?

*A helper for small farmers in the Po Valley, northern Italy (Piedmont, Lombardy, Veneto, Emilia-Romagna).*

**How to use it**
1. Run all cells in order (on Colab: *Runtime → Run all*).
2. In the map at the bottom **click on your field**: the coordinates are taken from the click.
3. If you want, choose last year's crop, whether you can irrigate and the results of your soil test.
4. Press **Compute**: after about 30 seconds the 5 most suitable crops and the advice appear.

**What happens behind the scenes**
- It downloads 20 years of daily climate for the point (NASA POWER, 2006-2025) and the soil (SoilGrids).
- For every crop and each of the 20 years it computes the same variables used to train the networks.
- It uses the **4 neural networks** (`modelli.joblib`, created by `1_training_networks_EN.ipynb`): yield, water, nitrogen, stability.
- The expected yield is the median of the 20 years; the "dry year" water is what is needed at least 1 year in 5.
- It ranks the crops and writes the advice with simple rules (rotations, pH, water).

**File needed:** `modelli.joblib`. If it is not in the notebook's folder (for example on Colab), it is downloaded automatically from the GitHub repository.

**On GitHub the interactive map is not shown:** at the bottom (section 7) there is an example already computed on a real field.

> ⚠️ Hackathon prototype: **this is not agronomic advice**. Talk to an agronomist before sowing.

## 0. Libraries

In [1]:
# subprocess and sys are used to install a library if it is missing.
import subprocess
import sys

# Try to import ipyleaflet (the map); if it is missing, install it with pip.
try:
    import ipyleaflet
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ipyleaflet"])

# The networks were saved with scikit-learn 1.8.0. If there is another version here (e.g. on Colab), we install the right one
# before importing it, so the networks load without problems (about 20 seconds, only the first time).
from importlib.metadata import version
if version("scikit-learn") != "1.8.0":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn==1.8.0"])

# On Google Colab interactive maps must be enabled; outside Colab this step is skipped.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

# io reads text as if it were a file; os checks whether a file exists; re finds pieces of text; time makes pauses.
import io
import os
import re
import time

# ipywidgets creates buttons, menus and boxes; we call it "widgets".
import ipywidgets as widgets

# joblib reads the file with the 4 networks.
import joblib

# numpy does maths on lists of numbers; pandas handles tables; requests downloads data from the internet.
import numpy as np
import pandas as pd
import requests

# sklearn (scikit-learn) is the networks' library: we need its version number.
import sklearn

# From the map library: the map, the pin, the small circles and the rectangle.
from ipyleaflet import CircleMarker, Map, Marker, Rectangle

# display shows tables and formatted text in the notebook; HTML writes text with bold and lists.
from IPython.display import HTML, display

## 1. The 4 networks and the crop tables
`modelli.joblib` contains the 4 networks, the median yield of every crop and the 33 lowland points used for training.
The tables below are written by hand (sources in `01_Datasets/tabelle/FONTI.md`).

In [2]:
# The project's GitHub repository, used to download the networks if they are missing.
REPO = "byterunnersAI/FarmerBud"

# If modelli.joblib is not in the folder (for example on Colab), download it from GitHub.
if not os.path.exists("modelli.joblib"):
    answer = requests.get(f"https://raw.githubusercontent.com/{REPO}/main/02_AI_Model/modelli.joblib", timeout=120)
    # If GitHub answers with an error, stop with a clear message.
    answer.raise_for_status()
    with open("modelli.joblib", "wb") as file:
        file.write(answer.content)

# Read the file with the 4 networks.
models = joblib.load("modelli.joblib")

# Networks saved with another scikit-learn version usually work, but it is better to know.
if models["sklearn_version"] != sklearn.__version__:
    print(f"⚠️ Networks created with scikit-learn {models['sklearn_version']}, here there is {sklearn.__version__}. "
          "Restart the session (Runtime → Restart session) and run all cells again.")

# Table of the 33 lowland points (name, coordinates, soil).
provinces = models["provinces"]

# The crops. For each one: name; season for the climate (start and end month, if start > end it begins the year before);
# growing degree-day base (°C); sowing date and cycle length (for water); optimal and limit pH; suitable soils.
CROPS = {
    "COMMEAT": dict(name="soft wheat", season=(10, 6), gdd_base=0, sowing="10-25", days=240, ph=(6.0, 7.0, 5.5, 8.5), soils="medium"),
    "WHEATD": dict(name="durum wheat", season=(10, 6), gdd_base=0, sowing="10-25", days=240, ph=(6.0, 7.0, 5.5, 8.5), soils="heavy, medium"),
    "BARLEY": dict(name="barley", season=(10, 6), gdd_base=0, sowing="10-15", days=235, ph=(6.5, 7.5, 6.0, 8.0), soils="medium"),
    "RAPE": dict(name="rapeseed", season=(9, 6), gdd_base=0, sowing="09-15", days=270, ph=(6.5, 7.6, 5.5, 8.0), soils="medium, light"),
    "MAIZE": dict(name="maize", season=(4, 9), gdd_base=10, sowing="04-10", days=160, ph=(5.0, 7.0, 4.5, 8.5), soils="medium"),
    "SOYA": dict(name="soybean", season=(5, 9), gdd_base=10, sowing="05-10", days=140, ph=(5.5, 6.5, 4.5, 8.4), soils="medium"),
    "SUNFLO": dict(name="sunflower", season=(4, 8), gdd_base=10, sowing="04-15", days=130, ph=(6.0, 7.5, 5.5, 8.0), soils="medium, light"),
    "SORGH": dict(name="sorghum", season=(5, 9), gdd_base=10, sowing="05-15", days=130, ph=(5.5, 7.5, 5.0, 8.0), soils="heavy, medium"),
    "RICE": dict(name="rice", season=(4, 9), gdd_base=10, sowing="04-20", days=150, ph=(5.5, 7.0, 4.5, 9.0), soils="heavy, medium, light"),
    "BEETS": dict(name="sugar beet", season=(3, 9), gdd_base=5, sowing="03-15", days=170, ph=(6.0, 6.8, 5.5, 8.5), soils="heavy, medium"),
}

# Last year's crop: the 10 crops plus alfalfa (which we do not predict, but it matters for rotation).
PREVIOUS_OPTIONS = [("don't know / other", None)] + [(c["name"], code) for code, c in CROPS.items()] + [("alfalfa", "LUCERNE")]

# Pairs (crop, previous year's crop) FORBIDDEN by the official guidelines, with the reason.
FORBIDDEN = {
    ("BEETS", "RAPE"): "beet cyst nematode (Heterodera schachtii), shared with brassicas",
    ("RAPE", "BEETS"): "beet cyst nematode (Heterodera schachtii)",
    ("RAPE", "SOYA"): "Sclerotinia, a disease shared by rapeseed, soybean and sunflower",
    ("RAPE", "SUNFLO"): "Sclerotinia, a disease shared by rapeseed, soybean and sunflower",
    ("SOYA", "RAPE"): "Sclerotinia, a disease shared by rapeseed, soybean and sunflower",
    ("SOYA", "SUNFLO"): "Sclerotinia, a disease shared by rapeseed, soybean and sunflower",
    ("SUNFLO", "RAPE"): "Sclerotinia, a disease shared by rapeseed, soybean and sunflower",
    ("SUNFLO", "SOYA"): "Sclerotinia, a disease shared by rapeseed, soybean and sunflower",
    ("WHEATD", "WHEATD"): "durum wheat cannot follow an autumn cereal (Fusarium)",
    ("WHEATD", "COMMEAT"): "durum wheat cannot follow an autumn cereal (Fusarium)",
    ("WHEATD", "BARLEY"): "durum wheat cannot follow an autumn cereal (Fusarium)",
    ("RAPE", "RAPE"): "same crop twice in a row not allowed",
    ("SOYA", "SOYA"): "same crop twice in a row not allowed",
    ("SUNFLO", "SUNFLO"): "same crop twice in a row not allowed",
    ("BEETS", "BEETS"): "nematode: sugar beet may return to the same field only after 3 years",
}

# Pairs allowed but with a WARNING, with the reason.
WARNINGS = {
    ("MAIZE", "MAIZE"): "maize after maize: allowed only once in 5 years; rotation alone keeps the corn rootworm under control",
    ("COMMEAT", "COMMEAT"): "same crop again allowed only once in 5 years (Fusarium and take-all)",
    ("COMMEAT", "BARLEY"): "after barley it counts as the same crop again (Fusarium)",
    ("COMMEAT", "WHEATD"): "after durum wheat it counts as the same crop again (Fusarium)",
    ("BARLEY", "BARLEY"): "same crop again allowed only once in 5 years",
    ("BARLEY", "COMMEAT"): "after wheat it counts as the same crop again",
    ("BARLEY", "WHEATD"): "after wheat it counts as the same crop again",
    ("SORGH", "SORGH"): "same crop again allowed only once in 5 years",
    ("BEETS", "LUCERNE"): "wireworms: better to wait 2 years after alfalfa",
    ("MAIZE", "LUCERNE"): "wireworms: break the alfalfa the summer before or right before sowing",
    ("WHEATD", "MAIZE"): "Fusarium: plough in the maize residues",
    ("WHEATD", "SORGH"): "Fusarium: plough in the sorghum residues",
}

# How much water "costs" in the ranking: relative-yield points removed per 100 mm of irrigation (team choice).
WATER_WEIGHT_IRRIGATED = 0.05
WATER_WEIGHT_RAINFED = 0.15

## 2. Download climate, elevation and soil for the point
The corrections are the same used to build the training data (`01_Datasets/s3_clima.py`): if they were different,
the networks would receive numbers they are not used to.
- **Elevation** from the Open-Meteo service: needed to correct the temperature and for ET0.
- **Climate** from NASA POWER: cells of about 50 km, temperature corrected for the elevation difference (calibrated on Open-Meteo).
- **ET0** (how much water the air "asks for") with the FAO-56 Penman-Monteith formula.
- **Soil** from SoilGrids (0-30 cm average). Towns have no data: we try a few nearby points and, if it does not
  answer, we use the soil of the nearest lowland point.

In [3]:
# Return the elevation (m) of a point, from the free Open-Meteo service.
def get_elevation(lat, lon):
    answer = requests.get("https://api.open-meteo.com/v1/elevation", params={"latitude": lat, "longitude": lon}, timeout=30)
    return answer.json()["elevation"][0]


# Daily ET0 (mm) with FAO-56, equation 6. Each line is a piece of the formula in the FAO manual.
def et0_penman_monteith(tmean, tmax, tmin, rh, u2, rs, lat, elevation, doy):
    # Saturation vapour pressure at temperature t (kPa).
    e0 = lambda t: 0.6108 * np.exp(17.27 * t / (t + 237.3))
    # Mean saturation and actual vapour pressure (from relative humidity).
    es = (e0(tmax) + e0(tmin)) / 2
    ea = rh / 100 * es
    # Slope of the vapour curve and psychrometric constant (depends on pressure, so on elevation).
    delta = 4098 * e0(tmean) / (tmean + 237.3) ** 2
    gamma = 0.000665 * 101.3 * ((293 - 0.0065 * elevation) / 293) ** 5.26
    # Radiation at the top of the atmosphere, from latitude and day of the year.
    phi = np.radians(lat)
    dr = 1 + 0.033 * np.cos(2 * np.pi * doy / 365)
    decl = 0.409 * np.sin(2 * np.pi * doy / 365 - 1.39)
    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(decl), -1, 1))
    ra = 24 * 60 / np.pi * 0.0820 * dr * (ws * np.sin(phi) * np.sin(decl) + np.cos(phi) * np.cos(decl) * np.sin(ws))
    # Net radiation: incoming (short wave) minus outgoing (long wave).
    rso = (0.75 + 2e-5 * elevation) * ra
    rnl = 4.903e-9 * ((tmax + 273.16) ** 4 + (tmin + 273.16) ** 4) / 2 * (0.34 - 0.14 * np.sqrt(ea)) * (1.35 * np.clip(rs / rso, 0, 1) - 0.35)
    rn = 0.77 * rs - rnl
    # The final Penman-Monteith formula; ET0 cannot be negative.
    et0 = (0.408 * delta * rn + gamma * 900 / (tmean + 273) * u2 * (es - ea)) / (delta + gamma * (1 + 0.34 * u2))
    return np.clip(et0, 0, None)


# Download daily climate from 1 September 2005 to 31 December 2025 from NASA POWER and correct it.
def download_climate(lat, lon, elevation):
    answer = requests.get("https://power.larc.nasa.gov/api/temporal/daily/point", timeout=180, params={
        "parameters": "T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN,RH2M,WS2M",
        "community": "AG", "latitude": lat, "longitude": lon,
        "start": "20050901", "end": "20251231", "format": "CSV"})
    text = answer.text
    # The file header contains the average elevation of the NASA POWER cell.
    cell_elevation = float(re.search(r"=\s*([\d.]+)\s*meters", text).group(1))
    # The real data start after the "-END HEADER-" line; -999 means missing data.
    daily = pd.read_csv(io.StringIO(text.split("-END HEADER-")[1].strip())).replace(-999, np.nan)
    # Build the date from year and day of the year (DOY).
    daily["date"] = pd.to_datetime(daily["YEAR"].astype(str) + daily["DOY"].astype(str).str.zfill(3), format="%Y%j")
    # Temperature correction: -0.56 °C + 4.2 °C for every 1000 m the cell is higher than the field.
    correction = -0.56 + (cell_elevation - elevation) * 0.64 * 6.5 / 1000
    for column in ["T2M", "T2M_MAX", "T2M_MIN"]:
        daily[column] = daily[column] + correction
    # Compute the ET0 of every day.
    daily["ET0"] = et0_penman_monteith(daily["T2M"], daily["T2M_MAX"], daily["T2M_MIN"], daily["RH2M"], daily["WS2M"],
                                       daily["ALLSKY_SFC_SW_DWN"], lat, elevation, daily["DOY"])
    return daily


# Query SoilGrids at a point and return pH, texture, carbon and nitrogen (0-30 cm average), or None.
def query_soilgrids(lat, lon):
    names = {"phh2o": "ph", "clay": "argilla_pct", "sand": "sabbia_pct", "silt": "limo_pct",
             "soc": "carbonio_organico_g_kg", "nitrogen": "azoto_g_kg"}
    # Thickness of each layer in cm: it is the weight in the average.
    depths = {"0-5cm": 5, "5-15cm": 10, "15-30cm": 15}
    try:
        answer = requests.get("https://rest.isric.org/soilgrids/v2.0/properties/query", timeout=60,
                              params=[("lon", round(lon, 4)), ("lat", round(lat, 4)), ("value", "mean")]
                              + [("property", p) for p in names] + [("depth", d) for d in depths])
        layers = answer.json()["properties"]["layers"]
    except Exception:
        return None
    soil = {}
    for layer in layers:
        # Pairs (thickness, value) of the layers that have data.
        values = [(depths[d["label"]], d["values"]["mean"]) for d in layer["depths"] if d["values"]["mean"] is not None]
        # No data (town or water): this point is no good.
        if not values:
            return None
        # Weighted average, divided by the SoilGrids "d_factor" to get the normal units.
        soil[names[layer["name"]]] = sum(w * v for w, v in values) / sum(w for w, _ in values) / layer["unit_measure"]["d_factor"]
    return soil


# Look for the soil at the point and, if missing, at 4 points about 2 km away; if nothing is found use the nearest lowland point.
def get_soil(lat, lon, nearest):
    for dlat, dlon in [(0, 0), (0.02, 0), (-0.02, 0), (0, 0.02), (0, -0.02)]:
        soil = query_soilgrids(lat + dlat, lon + dlon)
        if soil:
            return soil, "SoilGrids"
        # Short pause so we do not overload the service.
        time.sleep(1)
    columns = ["ph", "argilla_pct", "sabbia_pct", "limo_pct", "carbonio_organico_g_kg", "azoto_g_kg"]
    return nearest[columns].to_dict(), f"nearest lowland point ({nearest['comune_punto']})"

## 3. From daily climate to the network inputs
Each network wants its inputs **with the same names and the same meaning** they had in training.
- **Yield network:** 13 variables on the climate of that crop's season in that year.
- **Water network:** ET0 and rain from sowing to harvest (rain divided by 0.80, as in the training data).
- **Nitrogen network:** rain from October to February (above 300 mm water washes nitrogen away).
- **Stability network:** average and variability of the season climate over the 20 years.

The `predict` function avoids repeating code: it turns the crop into 0/1 columns, puts the columns in the right
order (`reindex`) and asks the network for the prediction.

In [4]:
# NASA POWER rain is about 80% of Open-Meteo's: for water we scale it up (as in training).
RAIN_RATIO = 0.80

# Climate years used to simulate the field.
YEARS = range(2006, 2026)


# The longest dry spell: how many days in a row with less than 1 mm of rain.
def longest_dry_spell(rain):
    best = current = 0
    for dry in (rain < 1.0):
        current = current + 1 if dry else 0
        best = max(best, current)
    return best


# The 13 climate variables for one crop and one harvest year (same as 01_Datasets/s3_clima.py).
def season_climate(daily, code, year):
    start_month, end_month = CROPS[code]["season"]
    # If the start month is after the end month, the season starts the year before (autumn crops).
    start = pd.Timestamp(year - 1 if start_month > end_month else year, start_month, 1)
    end = pd.Timestamp(year, end_month, 1) + pd.offsets.MonthEnd(0)
    s = daily[(daily["date"] >= start) & (daily["date"] <= end)]
    return dict(
        temp_media=s["T2M"].mean(), temp_max_media=s["T2M_MAX"].mean(), temp_min_media=s["T2M_MIN"].mean(),
        pioggia_mm=s["PRECTOTCORR"].sum(), et0_mm=s["ET0"].sum(),
        bilancio_idrico_mm=s["PRECTOTCORR"].sum() - s["ET0"].sum(),
        radiazione_MJ_m2=s["ALLSKY_SFC_SW_DWN"].sum(), umidita_media=s["RH2M"].mean(), vento_medio_m_s=s["WS2M"].mean(),
        gradi_giorno=np.clip(s["T2M"] - CROPS[code]["gdd_base"], 0, None).sum(),
        giorni_sopra_35=int((s["T2M_MAX"] > 35).sum()), giorni_gelo=int((s["T2M_MIN"] < 0).sum()),
        periodo_secco_max_giorni=longest_dry_spell(s["PRECTOTCORR"].values),
    )


# ET0 and rain from sowing to harvest, for the water network.
def cycle_water(daily, code, year):
    month, day = map(int, CROPS[code]["sowing"].split("-"))
    # Sowings from August on are autumn sowings: they happen the year before harvest.
    sowing = pd.Timestamp(year - 1 if month >= 8 else year, month, day)
    s = daily[(daily["date"] >= sowing) & (daily["date"] < sowing + pd.Timedelta(days=CROPS[code]["days"]))]
    return dict(et0_ciclo_mm=s["ET0"].sum(), pioggia_ciclo_mm=s["PRECTOTCORR"].sum() / RAIN_RATIO)


# Rain from 1 October to 28 February before harvest, for the nitrogen network.
def winter_rain(daily, year):
    s = daily[(daily["date"] >= pd.Timestamp(year - 1, 10, 1)) & (daily["date"] <= pd.Timestamp(year, 2, 28))]
    return s["PRECTOTCORR"].sum() / RAIN_RATIO


# Ask one of the 4 networks for a prediction; "rows" is a table with the inputs and the cod_coltura column.
def predict(network, rows):
    m = models[network]
    # Crop -> 0/1 columns; reindex puts the columns in training order and adds the missing ones as 0.
    X = pd.get_dummies(rows, columns=["cod_coltura"], dtype=float).reindex(columns=m["columns"], fill_value=0.0)
    # Same scale as in training, then the network predicts.
    return m["net"].predict(m["scaler"].transform(X))

## 4. The full computation for a field
For every crop:
1. **Yield:** the network predicts the relative yield in each of the 20 years; the expected yield is the median. In t/ha = relative yield × the crop's median yield.
2. **Water:** the network predicts irrigation in each of the 20 years; normal year = median, dry year = 80th percentile (needed at least 1 year in 5).
3. **Nitrogen:** the network receives the expected yield in t/ha and the farmer's choices.
4. **Stability:** the network predicts the coefficient of variation of the yield (%).

Then the **ranking**: score = relative yield − water weight × normal-year water / 100.
Crops forbidden by rotation, with pH outside the limits, or rice without irrigation go to the bottom, excluded from the top 5.

In [5]:
# Approximate distance in km between two points (fine for short distances).
def distance_km(lat1, lon1, lat2, lon2):
    return 111 * np.sqrt((lat1 - lat2) ** 2 + ((lon1 - lon2) * np.cos(np.radians(lat1))) ** 2)


# The full computation. "choices" holds the farmer's answers; "log" is the function that writes the messages.
def analyse_field(lat, lon, choices, log=print):
    # The nearest of the 33 lowland points used for training, and its distance.
    distances = distance_km(lat, lon, provinces["lat"], provinces["lon"])
    nearest = provinces.loc[distances.idxmin()]

    # Check 1: the point must be close to the training data (within 60 km).
    if distances.min() > 60:
        raise ValueError("The point is outside the Po Valley covered by the data: choose a field closer to the green circles.")

    log("Reading the elevation of the point...")
    elevation = get_elevation(lat, lon)
    # Check 2: it must be lowland. The training points are all below 350 m.
    if elevation > 400:
        raise ValueError(f"The point is at {elevation:.0f} m: it is not lowland. The networks only know fields below 400 m.")

    log("Looking for the soil on SoilGrids...")
    soil, soil_source = get_soil(lat, lon, nearest)
    # Check 3: at sea the elevation is exactly 0 and SoilGrids has no data.
    if elevation == 0 and soil_source != "SoilGrids":
        raise ValueError("The point seems to be at sea or in a lagoon: choose a field on land.")

    log("Downloading 20 years of climate from NASA POWER (about 15 seconds)...")
    daily = download_climate(lat, lon, elevation)
    # If the farmer entered their soil test, those values replace SoilGrids.
    if choices["lab"]:
        soil.update(choices["lab"])
        soil_source = "farmer's soil test + " + soil_source

    # The networks only know the soils of the 33 training points: a value outside that range is
    # brought back to the nearest limit (otherwise the networks "make things up"), and we tell the farmer.
    soil_columns = ["ph", "argilla_pct", "sabbia_pct", "limo_pct", "carbonio_organico_g_kg", "azoto_g_kg"]
    low, high = provinces[soil_columns].min(), provinces[soil_columns].max()
    clipped = [c for c in soil_columns if not low[c] <= soil[c] <= high[c]]
    network_soil = {c: float(np.clip(soil[c], low[c], high[c])) for c in soil_columns}

    # Soil and location: the same for all crops and all years.
    place = dict(network_soil, lat=lat, lon=lon, quota_m=elevation)
    # Median winter rain over the 20 years, for nitrogen.
    rain_winter = np.median([winter_rain(daily, year) for year in YEARS])

    log("The 4 neural networks are computing...")
    results = []
    for code in CROPS:
        # One row per year: season climate + soil and location + crop.
        years = pd.DataFrame([dict(season_climate(daily, code, year), **place, cod_coltura=code) for year in YEARS])

        # 1. Relative yield over the 20 years and expected yield (median).
        relative = np.median(predict("yield", years))
        t_ha = relative * models["crop_median_t_ha"][code]

        # 2. Water over the 20 years (in mm, never negative): normal year and dry year.
        water_rows = pd.DataFrame([dict(cycle_water(daily, code, year), cod_coltura=code) for year in YEARS])
        water = (predict("water", water_rows) * 100).clip(min=0)

        # 3. Nitrogen: a single row, with the expected yield and the farmer's choices.
        nitrogen_row = pd.DataFrame([dict(resa_t_ha=t_ha, pioggia_ott_feb_mm=rain_winter,
                                          dopo_medica=int(choices["previous"] == "LUCERNE"),
                                          sostanza_organica=choices["organic_matter"],
                                          residui_interrati=int(choices["residues"]), cod_coltura=code)])
        nitrogen = max(0.0, predict("nitrogen", nitrogen_row)[0] * 100)

        # 4. Stability: average climate, climate variability (_std), soil and location.
        st = models["stability"]
        stability_row = pd.DataFrame([dict(years[st["climate"]].mean(),
                                           **years[st["climate_spread"]].std().add_suffix("_std"),
                                           **place, cod_coltura=code)])
        cv = predict("stability", stability_row)[0] * 10

        # One results row for this crop.
        results.append(dict(code=code, crop=CROPS[code]["name"], relative=relative, t_ha=t_ha,
                            water_normal=np.median(water), water_dry=np.percentile(water, 80),
                            nitrogen=nitrogen, cv=cv))

    table = pd.DataFrame(results)
    # Score: relative yield counts most, water removes points (more if you cannot irrigate).
    weight = WATER_WEIGHT_IRRIGATED if choices["can_irrigate"] else WATER_WEIGHT_RAINFED
    table["score"] = table["relative"] - weight * table["water_normal"] / 100
    # Excluded crops: forbidden by rotation, pH outside the limits, rice if you cannot irrigate.
    table["excluded"] = [(code, choices["previous"]) in FORBIDDEN
                         or not CROPS[code]["ph"][2] <= soil["ph"] <= CROPS[code]["ph"][3]
                         or (code == "RICE" and not choices["can_irrigate"]) for code in table["code"]]
    # Sort: non-excluded first, then by score from highest.
    table = table.sort_values(["excluded", "score"], ascending=[True, False]).reset_index(drop=True)
    return table, soil, soil_source, elevation, clipped

## 5. Heatmap table and plain-language advice
Every column is coloured from **green** (better) to **red** (worse): for yield higher is better, for water,
nitrogen and variability lower is better. The advice comes from **rules** we wrote starting from the numbers,
not from a model that could make things up.

In [6]:
# Show the coloured table of the top 5 crops.
def show_table(table):
    top5 = table[~table["excluded"]].head(5)
    # A table with understandable column names and rounded numbers.
    shown = pd.DataFrame({
        "Crop": top5["crop"],
        "Expected yield (t/ha)": top5["t_ha"].round(1),
        "Yield vs average (%)": ((top5["relative"] - 1) * 100).round(0),
        "Water, normal year (mm)": top5["water_normal"].round(0),
        "Water, dry year (mm)": top5["water_dry"].round(0),
        "Nitrogen (kg N/ha)": top5["nitrogen"].round(0),
        "Yield variability (CV %)": top5["cv"].round(0),
    }).set_index("Crop")
    # Colours: RdYlGn goes from red to green (high = green); RdYlGn_r the other way round (low = green).
    # Yield in t/ha is not coloured: it cannot be compared across crops (sugar beet makes 60 t/ha, soybean 3).
    styled = (shown.style
              .background_gradient(cmap="RdYlGn", subset=[shown.columns[1]])
              .background_gradient(cmap="RdYlGn_r", subset=list(shown.columns[2:]))
              .format("{:.0f}").format("{:.1f}", subset=[shown.columns[0]]))
    display(styled)


# Readable names of the soil columns, for the warnings.
SOIL_NAMES = {"ph": "pH", "argilla_pct": "clay", "sabbia_pct": "sand", "limo_pct": "silt",
              "carbonio_organico_g_kg": "organic carbon", "azoto_g_kg": "nitrogen"}


# Write the advice in plain words, one per line.
def advice(table, soil, choices, clipped):
    tips = []
    # Warning if some soil value was outside those seen in training.
    if clipped:
        names = ", ".join(SOIL_NAMES[c] for c in clipped)
        tips.append(f"ℹ️ Some soil values ({names}) are outside those the networks saw in training: they used the closest value they know, so the results are less reliable.")
    previous = choices["previous"]
    for row in table[~table["excluded"]].head(5).itertuples():
        name = f"<b>{row.crop}</b>"
        # Yield: how much above or below the Po Valley average.
        percent = (row.relative - 1) * 100
        if abs(percent) < 1:
            tips.append(f"{name}: here it yields like the Po Valley average ({row.t_ha:.1f} t/ha).")
        else:
            tips.append(f"{name}: here it yields about {abs(percent):.0f}% {'more' if percent > 0 else 'less'} than the Po Valley average ({row.t_ha:.1f} t/ha).")
        # Water: warning if a dry year needs a lot of irrigation.
        if row.water_dry > 250:
            tips.append(f"{name}: in a dry summer like 2022 it would need about {row.water_dry:.0f} mm of irrigation: consider an alternative if water is scarce.")
        # Nitrogen: legumes do not need it.
        if row.code == "SOYA":
            tips.append(f"{name}: it is a legume: it needs no nitrogen fertiliser and leaves nitrogen in the soil for the next crop.")
        # Stability: warning if the yield changes a lot from year to year.
        if row.cv > 18:
            tips.append(f"{name}: the yield changes a lot from year to year (variability {row.cv:.0f}%): it is a riskier choice.")
        # Rotation: warnings about last year's crop.
        if (row.code, previous) in WARNINGS:
            tips.append(f"{name}: {WARNINGS[(row.code, previous)]}.")
        # pH outside the optimal range (but within the limits, otherwise it would be excluded).
        low, high = CROPS[row.code]["ph"][:2]
        if not low <= soil["ph"] <= high:
            tips.append(f"{name}: soil pH {soil['ph']:.1f}, outside the crop's optimum ({low}-{high}).")
    # The excluded crops and the reason.
    for row in table[table["excluded"]].itertuples():
        if (row.code, previous) in FORBIDDEN:
            reason = FORBIDDEN[(row.code, previous)]
        elif row.code == "RICE" and not choices["can_irrigate"]:
            reason = "without irrigation you cannot have a paddy field"
        else:
            reason = f"soil pH {soil['ph']:.1f} outside the crop's limits"
        tips.append(f"❌ <b>{row.crop}</b> excluded: {reason}.")
    # A bullet list in HTML.
    display(HTML("<ul>" + "".join(f"<li>{t}</li>" for t in tips) + "</ul>"))


# Compute and show everything for a field: header, table, advice and warning.
def show_results(lat, lon, choices):
    try:
        table, soil, soil_source, elevation, clipped = analyse_field(lat, lon, choices)
    except ValueError as problem:
        # The point is not valid (not lowland, at sea or far from the data): we explain why and stop.
        print("⚠️", problem)
        return
    # Header: coordinates, elevation and soil used.
    display(HTML(f"<h3>🌱 FarmerBud: the 5 most suitable crops for your field</h3>"
                 f"<p>{lat:.4f}, {lon:.4f} · {elevation:.0f} m · soil from {soil_source} · "
                 f"pH {soil['ph']:.1f} · clay {soil['argilla_pct']:.0f}% · sand {soil['sabbia_pct']:.0f}%</p>"))
    show_table(table)
    display(HTML("<h3>Advice</h3>"))
    advice(table, soil, choices, clipped)
    display(HTML("<p><i>⚠️ Hackathon prototype, this is not agronomic advice. Talk to an agronomist before sowing.</i></p>"))

## 6. The interface: map and questions for the farmer
- The **green circles** are the 33 lowland points used to train the networks; the **rectangle** is the area covered.
- **Click on your field**: the pin appears and the coordinates are written below the map.
- The questions are optional. For the soil test leave empty what you do not know.

In [7]:
# The map, centred on the Po Valley; you cannot zoom out too far.
field_map = Map(center=(45.1, 10.3), zoom=7, min_zoom=7, scroll_wheel_zoom=True, layout=widgets.Layout(height="450px"))

# The rectangle showing the area covered by the data.
field_map.add(Rectangle(bounds=((44.0, 7.3), (46.0, 12.8)), color="green", fill_opacity=0.03, weight=1))

# One green circle for each of the 33 lowland points.
for p in provinces.itertuples():
    field_map.add(CircleMarker(location=(p.lat, p.lon), radius=4, color="green", fill_opacity=0.7, weight=1))

# The field pin: not there at first, it appears at the first click.
pin = Marker(location=(45.1, 10.3), draggable=False)

# The chosen point: none at the start.
chosen = {"lat": None, "lon": None}

# Text under the map with the coordinates.
coordinates = widgets.HTML("👆 Click on the map to choose your field.")


# This function is called by the map at every interaction (click, mouse move...).
def on_map_click(**event):
    # We only care about clicks.
    if event.get("type") != "click":
        return
    # The click coordinates: [latitude, longitude].
    lat, lon = event["coordinates"]
    chosen["lat"], chosen["lon"] = lat, lon
    # Move the pin to the click and add it to the map if it was not there.
    pin.location = (lat, lon)
    if pin not in field_map.layers:
        field_map.add(pin)
    coordinates.value = f"📍 Field chosen: <b>{lat:.4f}, {lon:.4f}</b>"


# Connect the function to the map.
field_map.on_interaction(on_map_click)

# The questions for the farmer (all optional).
style = {"description_width": "260px"}
previous_crop = widgets.Dropdown(options=PREVIOUS_OPTIONS, description="Last year's crop", style=style)
can_irrigate = widgets.Checkbox(value=True, description="I can irrigate (well, water board)", style=style)
organic_matter = widgets.Dropdown(options=[("normal / don't know", 0), ("low", -1), ("high", 1)],
                                  description="Soil organic matter", style=style)
residues = widgets.Checkbox(value=False, description="I plough in straw or stalks of the previous crop", style=style)

# Soil test: text boxes, empty = "I don't know".
lab_ph = widgets.Text(description="pH", placeholder="e.g. 7.2", style=style)
lab_clay = widgets.Text(description="Clay (%)", placeholder="e.g. 25", style=style)
lab_sand = widgets.Text(description="Sand (%)", placeholder="e.g. 30", style=style)
lab_silt = widgets.Text(description="Silt (%)", placeholder="e.g. 45", style=style)
lab_om = widgets.Text(description="Organic matter (%)", placeholder="e.g. 2.0", style=style)
lab_n = widgets.Text(description="Total nitrogen (g/kg)", placeholder="e.g. 1.4", style=style)

# The button and the area where the results appear.
button = widgets.Button(description="Compute", button_style="success", icon="seedling")
results_area = widgets.Output()


# Read a number from a text box; accepts a comma; if empty returns None.
def read_number(box):
    text = box.value.strip().replace(",", ".")
    return float(text) if text else None


# Collect the farmer's answers in a dictionary.
def read_choices():
    lab = {}
    for key, box in [("ph", lab_ph), ("argilla_pct", lab_clay), ("sabbia_pct", lab_sand), ("limo_pct", lab_silt), ("azoto_g_kg", lab_n)]:
        if read_number(box) is not None:
            lab[key] = read_number(box)
    # The lab gives organic matter in %; the networks want organic carbon in g/kg: OM ÷ 1.724 × 10.
    if read_number(lab_om) is not None:
        lab["carbonio_organico_g_kg"] = read_number(lab_om) / 1.724 * 10
    return {"previous": previous_crop.value, "can_irrigate": can_irrigate.value,
            "organic_matter": organic_matter.value, "residues": residues.value, "lab": lab}


# What happens when "Compute" is pressed.
def on_button_click(_):
    # Everything written goes into the results area, under the button.
    with results_area:
        # Clear the previous results.
        results_area.clear_output()
        if chosen["lat"] is None:
            print("First click on the map to choose the field.")
            return
        show_results(chosen["lat"], chosen["lon"], read_choices())


# Connect the function to the button.
button.on_click(on_button_click)

# Stack everything in a column and show it: map, coordinates, questions, soil test, button, results.
display(widgets.VBox([
    widgets.HTML("<h2>🌱 FarmerBud</h2>"),
    field_map, coordinates,
    widgets.HTML("<b>Your field (optional)</b>"),
    previous_crop, can_irrigate, organic_matter, residues,
    widgets.HTML("<b>Soil test (optional, leave empty what you don't know)</b>"),
    lab_ph, lab_clay, lab_sand, lab_silt, lab_om, lab_n,
    button, results_area,
]))

## 7. Example already computed (without the map)
On GitHub the interactive map is not shown, so here FarmerBud runs on an **example field** in the Cremona plain,
near Pizzighettone (45.15 N, 9.95 E). Last year the farmer grew **maize**, can irrigate and ploughs in the stalks.
You can change the coordinates and choices and run the cell again.

In [8]:
# The example farmer's choices: maize last year, irrigation yes, normal organic matter, stalks ploughed in, no soil test.
example_choices = {"previous": "MAIZE", "can_irrigate": True, "organic_matter": 0, "residues": True, "lab": {}}

# Compute and show the results for the example field.
show_results(45.15, 9.95, example_choices)

Reading the elevation of the point...


Looking for the soil on SoilGrids...


The 4 neural networks are computing...


,Expected yield (t/ha),Yield vs average (%),"Water, normal year (mm)","Water, dry year (mm)",Nitrogen (kg N/ha),Yield variability (CV %)
Crop,,,,,,
soft wheat,5.9,4,20,35,195,13
barley,5.3,-0,3,12,157,15
soybean,3.4,6,218,286,0,16
rapeseed,2.8,-6,6,13,176,17
durum wheat,5.2,-6,18,33,194,16
